In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import random
import pandas as pd

In [ ]:
def outlier_removal(df, column):

    df[column] = df[column].replace(-1, np.nan)

    r = df[column].dropna().to_numpy()
    
    if r.size == 0:
        print("Coluna não contém valores suficientes para análise.")
        return df

    r_max = np.max(r) 
    r = r / r_max  

    perc_min = []
    p_min = np.linspace(0.1, 2, 20)
    for i in p_min:
        perc_min.append(np.percentile(r, i))
    diff_perc_min = np.diff(perc_min)
    index_min = np.argmax(diff_perc_min)  
    thres_min = np.mean(perc_min[index_min:index_min + 2])

    perc_max = []
    p_max = np.linspace(98, 100, 20)
    for i in p_max:
        perc_max.append(np.percentile(r, i))
    diff_perc_max = np.diff(perc_max)
    index_max = np.argmax(diff_perc_max)  
    thres_max = np.mean(perc_max[index_max:index_max + 2])

    r_filtered = np.where((r < thres_min) | (r > thres_max), np.nan, r)

    r_filtered = r_filtered * r_max  

    df_filtered = df.copy()
    df_filtered.loc[~df[column].isna(), column] = r_filtered

    return df_filtered

def generate_missing_data(df, porcentagem):
    df_copy = df.copy()
    df_copy = outlier_removal(df, column='Throughput') # Removing outliers
    quantidade = (porcentagem * len(df_copy) / 100)
    indices_substituir = random.sample(df_copy.index.tolist(), round(quantidade))
    df_copy.loc[indices_substituir, 'Throughput'] = np.nan
    return df_copy

def rolling_imputation_analysis(df, original_df, missing_percentage, window_size=3, center=False, loop=False, method='median'):
    # Ensure the 'Timestamp' column is in datetime format and set it as the index
    if 'Timestamp' in df.columns:
        df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
        original_df['Timestamp'] = pd.to_datetime(original_df['Timestamp'], errors='coerce')
        
        df = df.set_index('Timestamp')
        original_df = original_df.set_index('Timestamp')

    # Check for any remaining issues with the index format
    if not isinstance(df.index, pd.DatetimeIndex):
        print("The index is not a DatetimeIndex. Please ensure the 'Timestamp' column is in proper datetime format.")
        return df  # Return the original DataFrame if the conversion fails
    
    df = df.sort_index()
    original_df = original_df.sort_index()

    # Create a copy of df for imputation
    df_imputed = df.copy()

    # Identify missing indices BEFORE imputation
    missing_indices = df[df['Throughput'].isnull()].index
    
    # Apply rolling imputation based on the chosen method
    if method == 'median':
        if loop:
            previous_na_count = df_imputed['Throughput'].isna().sum()
            while df_imputed['Throughput'].isna().any():
                df_imputed['Throughput'] = df_imputed['Throughput'].fillna(
                    df_imputed['Throughput'].rolling(window=window_size, center=center, min_periods=1).median()
                )
                current_na_count = df_imputed['Throughput'].isna().sum()
                if current_na_count >= previous_na_count:
                    break  # No progress made, so exit the loop
                previous_na_count = current_na_count
            # global_median = df_imputed['Throughput'].median()
            # df_imputed['Throughput'] = df_imputed['Throughput'].fillna(global_median)

        else:
            df_imputed['Throughput'] = df_imputed['Throughput'].fillna(
                df_imputed['Throughput'].rolling(window=window_size, center=center, min_periods=1).median()
            )
            # global_median = df_imputed['Throughput'].median()
            # df_imputed['Throughput'] = df_imputed['Throughput'].fillna(global_median)

    elif method in ['average', 'mean']:
        if loop:
            previous_na_count = df_imputed['Throughput'].isna().sum()
            while df_imputed['Throughput'].isna().any():
                df_imputed['Throughput'] = df_imputed['Throughput'].fillna(
                    df_imputed['Throughput'].rolling(window=window_size, center=center, min_periods=1).mean()
                )
                current_na_count = df_imputed['Throughput'].isna().sum()
                if current_na_count >= previous_na_count:
                    break  # No progress made, so exit the loop
                previous_na_count = current_na_count
            # global_mean = df_imputed['Throughput'].mean()
            # df_imputed['Throughput'] = df_imputed['Throughput'].fillna(global_mean)
        else:
            df_imputed['Throughput'] = df_imputed['Throughput'].fillna(
                df_imputed['Throughput'].rolling(window=window_size, center=center, min_periods=1).mean()
            )
            # global_mean = df_imputed['Throughput'].mean()
            # df_imputed['Throughput'] = df_imputed['Throughput'].fillna(global_mean)

    else:
        print("Invalid rolling method")
        return

    # Print the imputed 'Throughput' series for verification
    print(df_imputed['Throughput'])

    # Calculate RMSE for imputed values only
    if df_imputed['Throughput'].isnull().any():
        rmse = -10  # Indica que ainda há valores ausentes após a imputação
    else:
        # Calculate errors at imputed positions
        errors = original_df.loc[missing_indices, 'Throughput'] - df_imputed.loc[missing_indices, 'Throughput']
        rmse = np.sqrt((errors ** 2).mean()) / 1000000
    print(f"RMSE for imputed values: {rmse}")

    result = {
        "Missing Percentage": missing_percentage,
        "Rolling Method": method,
        "Window Size": window_size,
        "Is on loop?": loop,
        "Center": center,
        "RMSE": rmse
    }
    return result, df_imputed



def plot_rmses_result_rolling(results_list):
    # Convert the results list into a DataFrame for easy manipulation
    df_original = pd.DataFrame(results_list)
    df = df_original.copy()

    # Sort and extract unique values for plotting
    missing_percentages = sorted(df['Missing Percentage'].unique())
    methods = df['Rolling Method'].unique()
    loop_options = df['Is on loop?'].unique()
    center_options = df['Center'].unique()

    # Set up the figure
    fig, ax = plt.subplots(figsize=(15, 10))

    # Define bar width and initial x-axis position for groups
    x = np.arange(len(missing_percentages))
    width = 0.1  # Width of each bar for clear distinction

    # Loop through each combination of method, loop, and center to plot RMSE
    bar_position = 0  # Keeps track of the position of each bar set
    for method in methods:
        for loop in loop_options:
            for center in center_options:
                # Filter DataFrame for the current combination of parameters
                df_filtered = df[(df['Rolling Method'] == method) &
                                 (df['Is on loop?'] == loop) &
                                 (df['Center'] == center)]
                
                # RMSE values for each missing percentage in the current parameter combination
                rmse_values = [
                    df_filtered[df_filtered['Missing Percentage'] == pct]['RMSE'].values[0]
                    if not df_filtered[df_filtered['Missing Percentage'] == pct].empty else np.nan
                    for pct in missing_percentages
                ]

                # Label for each bar group
                label = f'{method}, Loop={loop}, Center={center}'
                
                # Plot bars for the current parameter combination
                ax.bar(x + bar_position * width, rmse_values, width, label=label)
                bar_position += 1  # Update position for the next set of bars

    # Labeling and formatting
    ax.set_xlabel('Missing Percentage')
    ax.set_ylabel('RMSE')
    ax.set_title('RMSE by Missing Percentage and Rolling Method Combinations')
    ax.set_xticks(x + width * (bar_position - 1) / 2)
    ax.set_xticklabels([f'{pct}%' for pct in missing_percentages])
    ax.legend(title='Method, Loop, Center', bbox_to_anchor=(1.05, 1), loc='upper left')

    plt.tight_layout()
    plt.show()


In [ ]:
#Using the longest interval among 07-07-2023 datasets
df = pd.read_csv("../datasets/throughput/07-07-2024/longest interval/treated cubic esmond data ap-rs 07-03-2023_longest_interval.csv")

In [ ]:
df_missing10 = generate_missing_data(df, 10)
df_missing20 = generate_missing_data(df, 20)
df_missing30 = generate_missing_data(df, 30)

In [ ]:
dfs_missing = [df_missing10, df_missing20, df_missing30]
rolling_results = []

In [ ]:
missing_percentage = 10
for missing in dfs_missing:
    result, _ = rolling_imputation_analysis(missing, df, missing_percentage, center=False, loop=False, method='mean')
    rolling_results.append(result)

    result2, _ = rolling_imputation_analysis(missing, df, missing_percentage, center=True, loop=False, method='mean')
    rolling_results.append(result2)

    result3, _ = rolling_imputation_analysis(missing, df, missing_percentage, center=True, loop=True, method='mean')
    rolling_results.append(result3)

    result4, _ = rolling_imputation_analysis(missing, df, missing_percentage, center=False, loop=True, method='mean')
    rolling_results.append(result4)

    result5, _ = rolling_imputation_analysis(missing, df, missing_percentage, center=False, loop=False, method='median')
    rolling_results.append(result5)

    result6, _ = rolling_imputation_analysis(missing, df, missing_percentage, center=True, loop=False, method='median')
    rolling_results.append(result6)

    result7, _ = rolling_imputation_analysis(missing, df, missing_percentage, center=True, loop=True, method='median')
    rolling_results.append(result7)

    result8, _ = rolling_imputation_analysis(missing, df, missing_percentage, center=False, loop=True, method='median')
    rolling_results.append(result8)

    missing_percentage = missing_percentage + 10

In [ ]:
plot_rmses_result_rolling(rolling_results)